# Exploración de datos

**Minería de Datos II · ISTEA · Primera entrega - Prof. Diego Mosquera**

Lectua de datos con PySpark
cuántas filas tienen, qué columnas traen nulos, si hay duplicados y qué problemas de calidad aparecen.

No limpio ni guardo, solo analizo y, lo que encuentre acá, será usado como base para diseñar las capas bronze y silver

## Instalo PySpark

Igual que en el notebook de clase, luego lo comento para no instalar en la próxima corrida.

In [12]:
#!pip -q install pyspark

## Obtengo los datos
Realizo la clonación del repositorio del proyecto y descomprimo el dataset en la ruta `/content/datos`.
Tampoco modifico al zip original: es mi zona Landing (es decir,datos crudos inmutables).

In [13]:
!git clone -q https://github.com/damiann8n/cloud-provider-analytics.git
!unzip -q -o cloud-provider-analytics/data/cloud_provider_challenge_dataset_v1.zip -d /content/datos
!ls /content/datos/datalake/landing

fatal: destination path 'cloud-provider-analytics' already exists and is not an empty directory.
billing_monthly.csv    nps_surveys.csv	    usage_events_stream
customers_orgs.csv     resources.csv	    users.csv
marketing_touches.csv  support_tickets.csv


In [14]:
#Guardo la ruta en una variable para no repetirla cada vez que deba leerla
LANDING = "/content/datos/datalake/landing"

## Sesión de Spark

- `local[*]`: para usar todos los núcleos de la máquina de Colab.
- `shuffle.partitions = 8`: el dataset es chico, no necestio hacer uso de las 200 particiones por defecto de Spark

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("exploracion-cloud-provider")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Mi Versión de Spark:", spark.version)

Mi Versión de Spark: 4.0.4


## Lecuta de los archivos .csv

Leo los 7 archivos CSV. No haré uso de inferSchema, así visualizo los valores tal cual vienen y evito que Spark los transforme. Luego, en bronze, definiré un esquema explícito, ajustándome a lo solicitado en el trabajo prácito.

In [16]:
orgs      = spark.read.option("header", True).csv(f"{LANDING}/customers_orgs.csv")
users     = spark.read.option("header", True).csv(f"{LANDING}/users.csv")
resources = spark.read.option("header", True).csv(f"{LANDING}/resources.csv")
tickets   = spark.read.option("header", True).csv(f"{LANDING}/support_tickets.csv")
marketing = spark.read.option("header", True).csv(f"{LANDING}/marketing_touches.csv")
nps       = spark.read.option("header", True).csv(f"{LANDING}/nps_surveys.csv")
billing   = spark.read.option("header", True).csv(f"{LANDING}/billing_monthly.csv")

### Registros x archivo

In [17]:
print("customers_orgs   :", orgs.count())
print("users            :", users.count())
print("resources        :", resources.count())
print("support_tickets  :", tickets.count())
print("marketing_touches:", marketing.count())
print("nps_surveys      :", nps.count())
print("billing_monthly  :", billing.count())

customers_orgs   : 80
users            : 800
resources        : 400
support_tickets  : 1000
marketing_touches: 1500
nps_surveys      : 92
billing_monthly  : 240


### Visualización rápida de cada tabla

In [18]:
orgs.show(5, truncate=False)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |True         |2025-05-26 |rep_c    |churned        |partner         |NULL     |
|org_pbhsahxt|Nova Tech 1    |Education    |us-east   |standard |False        |2025-06-16 |rep_d    |lead           |event           |-3.0     |
|org_hv3a3zmf|Gamma Data 2   |Media        |eu-central|pro      |True         |2025-05-12 |rep_c    |active         |organic         |4.0      |
|org_8mdd4v30|Apex Data 3    |Manufacturing|sa-east   |standard |False        |2025-05-30 |rep_b    |active         |ads          

In [19]:
billing.show(5, truncate=False)

+------------+------------+----------+--------+-------+------+--------+--------------------+
|invoice_id  |org_id      |month     |subtotal|credits|taxes |currency|exchange_rate_to_usd|
+------------+------------+----------+--------+-------+------+--------+--------------------+
|inv_nym31sk0|org_xaji0y6d|2025-06-01|1133.34 |NULL   |238.0 |USD     |0.97684             |
|inv_540sejok|org_xaji0y6d|2025-07-01|619.17  |NULL   |130.03|USD     |1.00331             |
|inv_ilp4vn2v|org_xaji0y6d|2025-08-01|540.79  |NULL   |113.57|ARS     |0.0016              |
|inv_27nls4sz|org_pbhsahxt|2025-06-01|520.91  |NULL   |109.39|ARS     |0.00147             |
|inv_xq1k1qrr|org_pbhsahxt|2025-07-01|909.56  |NULL   |191.01|EUR     |1.09929             |
+------------+------------+----------+--------+-------+------+--------+--------------------+
only showing top 5 rows


In [20]:
tickets.show(5, truncate=False)

+------------+------------+------------+--------+----------+-----------+----+------------+
|ticket_id   |org_id      |category    |severity|created_at|resolved_at|csat|sla_breached|
+------------+------------+------------+--------+----------+-----------+----+------------+
|tkt_zjrumxqw|org_x7eedtjv|performance |low     |2025-07-02|2025-07-08 |4.0 |False       |
|tkt_3b4nthzm|org_gv0e38da|security    |low     |2025-07-21|2025-07-22 |NULL|False       |
|tkt_32hzs32d|org_9mx2x18h|availability|low     |2025-07-03|2025-07-05 |3.0 |False       |
|tkt_1jqbexgf|org_1swjckjl|usability   |critical|2025-07-18|2025-07-22 |4.0 |False       |
|tkt_vnnvbzw5|org_trp0j43d|integration |low     |2025-07-26|2025-07-31 |3.0 |False       |
+------------+------------+------------+--------+----------+-----------+----+------------+
only showing top 5 rows


## Valores nulos

Pequeña función con for... para recorrer las columnas y contar los nulos.
Muestro valores absolutos y relativos, para interpretar mejor la cantidad de datos faltantes.

In [21]:
def contar_nulos(df, nombre):
    total = df.count()
    print(f"--- {nombre} ({total} filas)")
    for columna in df.columns:
        nulos = df.filter(F.col(columna).isNull()).count()
        if nulos > 0:
            print(f"   {columna}: {nulos} nulos ({nulos / total:.1%})")

In [22]:
contar_nulos(orgs, "customers_orgs")
contar_nulos(users, "users")
contar_nulos(resources, "resources")
contar_nulos(tickets, "support_tickets")
contar_nulos(marketing, "marketing_touches")
contar_nulos(nps, "nps_surveys")
contar_nulos(billing, "billing_monthly")

--- customers_orgs (80 filas)
   nps_score: 11 nulos (13.8%)
--- users (800 filas)
   last_login: 139 nulos (17.4%)
--- resources (400 filas)
   tags_json: 83 nulos (20.8%)
--- support_tickets (1000 filas)
   resolved_at: 240 nulos (24.0%)
   csat: 254 nulos (25.4%)
--- marketing_touches (1500 filas)
--- nps_surveys (92 filas)
   nps_score: 19 nulos (20.7%)
   comment: 10 nulos (10.9%)
--- billing_monthly (240 filas)
   credits: 137 nulos (57.1%)


Se deben analizar los nulos "esperables" (aalguna session, por ejemplo) de aquellos valores faltantes que no deberían estar ausentes.

## Visualización de duplicados por clave
no deberían existir, y la cantidad de claves tiene que ser igual a la cantidad de registros:

In [23]:
print("orgs      duplicados:", orgs.count()      - orgs.select("org_id").distinct().count())
print("users     duplicados:", users.count()     - users.select("user_id").distinct().count())
print("resources duplicados:", resources.count() - resources.select("resource_id").distinct().count())
print("tickets   duplicados:", tickets.count()   - tickets.select("ticket_id").distinct().count())
print("billing   duplicados:", billing.count()   - billing.select("invoice_id").distinct().count())

orgs      duplicados: 0
users     duplicados: 0
resources duplicados: 0
tickets   duplicados: 0
billing   duplicados: 0


## Eventos (JSONL)

Los eventos me llegan repartidos en 120 archivos, simulando la llegada que se produciría en micro lotes. Los leo a todos juntos, en modo exploratorio.
Más adelante, en la implementación, los procesaré con streaming estructurado:

In [24]:
eventos = spark.read.json(f"{LANDING}/usage_events_stream")

eventos.printSchema()
print("Cantidad de eventos:", eventos.count())

root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

Cantidad de eventos: 43200


Spark está infiriendo a value` como `string` y no como dato numérico. En algunos casos el número es 133.00  y en otros como texto 133.00.
Estoy frente al problema de los tipos ambiguos que se mencionan en la consigna: en silver deben ser convertidos a valores numéricos.

### versión 1 vs versión 2

La consigna dice que a mitad del período aparecen los campos carbon_kg y genai_tokens. Paso a verificarlo:

In [25]:
eventos.groupBy("schema_version").agg(
    F.count("*").alias("eventos"),
    F.min("timestamp").alias("desde"),
    F.max("timestamp").alias("hasta"),
    F.count("carbon_kg").alias("con_carbon_kg"),
    F.count("genai_tokens").alias("con_genai_tokens"),
).orderBy("schema_version").show(truncate=False)

+--------------+-------+--------------------+--------------------+-------------+----------------+
|schema_version|eventos|desde               |hasta               |con_carbon_kg|con_genai_tokens|
+--------------+-------+--------------------+--------------------+-------------+----------------+
|1             |10800  |2025-07-03T00:02:00Z|2025-07-17T23:56:00Z|0            |0               |
|2             |32400  |2025-07-18T00:01:00Z|2025-08-31T23:58:00Z|32400        |3132            |
+--------------+-------+--------------------+--------------------+-------------+----------------+



F.count(columna) cuenta solo los valores **no nulos**. Veo que la v1 no trae carbon_kg ni genai_tokens, y que la v2 empieza el 18/07/2025.

### Revisar si el event_id es o no único

In [26]:
total_eventos = eventos.count()
ids_distintos = eventos.select("event_id").distinct().count()

print("Eventos:", total_eventos)
print("event_id distintos:", ids_distintos)

Eventos: 43200
event_id distintos: 43200


No encuentro duplicados en la raw data, pero voy a deduplicar en el streaming, para que al re ejecutar no se carguen dos veces el mismo evento (idempotencia):

### Nulos en los eventos

In [27]:
contar_nulos(eventos, "usage_events")

--- usage_events (43200 filas)
   carbon_kg: 10800 nulos (25.0%)
   genai_tokens: 40068 nulos (92.8%)
   unit: 2075 nulos (4.8%)
   value: 877 nulos (2.0%)


carbon_kg y genai_tokens tienen muchos valores nulos porque no existen en la v1 y genai_tokens solo aplica al servicio genai.
En cambio, los nulos en valu y unit son problemas de calidad en la imputación de los datos.

### Análisis: servicios, regiones y métricas

In [28]:
eventos.groupBy("service").count().orderBy(F.desc("count")).show()
eventos.groupBy("region").count().orderBy(F.desc("count")).show()
eventos.groupBy("metric", "unit").count().orderBy("metric").show()

+----------+-----+
|   service|count|
+----------+-----+
|   compute|12498|
|   storage| 7614|
|  database| 7419|
|networking| 6921|
| analytics| 4590|
|     genai| 4158|
+----------+-----+

+------------+-----+
|      region|count|
+------------+-----+
|ap-northeast| 7773|
|     us-east| 7566|
|     us-west| 6384|
|     sa-east| 6270|
|    ap-south| 5943|
|     eu-west| 4854|
|  eu-central| 4410|
+------------+-----+

+----------------+--------+-----+
|          metric|    unit|count|
+----------------+--------+-----+
|       cpu_hours|   hours|10187|
|       cpu_hours|    NULL|  505|
|        requests|   count|18596|
|        requests|    NULL|  916|
|storage_gb_hours|gb_hours|12342|
|storage_gb_hours|    NULL|  654|
+----------------+--------+-----+



### Revisando los costos negativos y valores extremos

La consigna indcia la presencia de costos negativos y spikes (picos muy altos).

In [29]:
negativos = eventos.filter(F.col("cost_usd_increment") < 0).count()
print("Eventos con costo negativo:", negativos)

eventos.select("cost_usd_increment").describe().show()

Eventos con costo negativo: 216
+-------+------------------+
|summary|cost_usd_increment|
+-------+------------------+
|  count|             43200|
|   mean|3.4128235601851857|
| stddev|7.9236209122554495|
|    min|         -154.4608|
|    max|          317.4308|
+-------+------------------+



In [30]:
#Percentiles: el 99% de los eventos cuesta menos que el valor p99.
#Si el máximo está muy lejos del p99, hay valores extremos (outliers).
percentiles = eventos.approxQuantile("cost_usd_increment", [0.5, 0.99], 0.001)
print("Mediana (p50):", percentiles[0])
print("Percentil 99 :", percentiles[1])

Mediana (p50): 1.0011
Percentil 99 : 16.4675


El máximo de 317 USD supera por casi 20 veces el p99 17 USD: esto me confirma la presencia de valores atípicos. Luego analizaré qué método utilizo con silver.

### Revisar si los eventos llegan ordenados:

Leo unarchivo y veo el rango de fechas que expone;

In [31]:
un_archivo = spark.read.json(f"{LANDING}/usage_events_stream/events_part_0000.jsonl")

un_archivo.select(F.min("timestamp"), F.max("timestamp")).show(truncate=False)
un_archivo.select("timestamp").show(5, truncate=False)

+--------------------+--------------------+
|min(timestamp)      |max(timestamp)      |
+--------------------+--------------------+
|2025-07-03T00:08:00Z|2025-08-31T16:32:00Z|
+--------------------+--------------------+

+--------------------+
|timestamp           |
+--------------------+
|2025-08-17T01:55:00Z|
|2025-07-16T11:20:00Z|
|2025-08-18T18:52:00Z|
|2025-07-29T03:32:00Z|
|2025-08-11T15:19:00Z|
+--------------------+
only showing top 5 rows


Un solo archivo trae eventos de todo el período julio/agosto y se observa que están desordenados.
En streaming llegarán datos tardíos, late data: por eso haré uso de watermark.

## Chequeos de calidad

### Facturación: tipo de cambio por moneda

Una factura en USD debería tener `exchange_rate_to_usd = 1`. Lo reviso (consultar al profesor)

In [33]:
billing.groupBy("currency").agg(
    F.count("*").alias("facturas"),
    F.min("exchange_rate_to_usd").alias("fx_min"),
    F.max("exchange_rate_to_usd").alias("fx_max"),
).show()

+--------+--------+-------+-------+
|currency|facturas| fx_min| fx_max|
+--------+--------+-------+-------+
|     ARS|      51|0.00133|0.00162|
|     EUR|      29| 0.9981|1.19808|
|     USD|     160|0.85463|1.11791|
+--------+--------+-------+-------+



Las facturas en USD tienen tipos de cambio entre 0,85 y 1,12: es un error de los datos, ya que (por ejemplo) el euro o dolar de junio debería ser el mismo para todos los clientes, pero no se observa ese comportamiento. Es probable que el dataset sintético haya generado ruido en estos campos, distorsionando el comportamiento de los datos reales.

In [34]:
print("Subtotales negativos:", billing.filter(F.col("subtotal").cast("double") < 0).count())
print("Créditos nulos      :", billing.filter(F.col("credits").isNull()).count())

Subtotales negativos: 13
Créditos nulos      : 137


### Satisfacción: CSAT, con escala de 1 a 5

In [35]:
tickets.groupBy("csat").count().orderBy("csat").show()

+----+-----+
|csat|count|
+----+-----+
|NULL|  254|
| 0.0|   11|
| 1.0|   45|
| 2.0|  127|
| 3.0|  242|
| 4.0|  197|
| 5.0|   95|
| 6.0|   28|
| 7.0|    1|
+----+-----+



Analizo valores 0, 6 y 7, que están fuera de la citada escala, los trataré como datos no válidos.

### NPS, con escala de −100 a 100

In [36]:
orgs.select(F.col("nps_score").cast("double").alias("nps_score")).describe().show()
nps.select(F.col("nps_score").cast("double").alias("nps_score")).describe().show()

+-------+------------------+
|summary|         nps_score|
+-------+------------------+
|  count|                69|
|   mean|18.666666666666668|
| stddev|26.071545604280512|
|    min|             -38.0|
|    max|             101.0|
+-------+------------------+

+-------+------------------+
|summary|         nps_score|
+-------+------------------+
|  count|                73|
|   mean|26.164383561643834|
| stddev|15.893756372507255|
|    min|             -16.0|
|    max|              68.0|
+-------+------------------+



### Análisis de tickets abiertos, críticos y con SLA incumplido:

In [37]:
print("Tickets abiertos (sin resolved_at):", tickets.filter(F.col("resolved_at").isNull()).count())
print("Tickets críticos                  :", tickets.filter(F.col("severity") == "critical").count())
print("Tickets con SLA incumplido        :", tickets.filter(F.col("sla_breached") == "True").count())

Tickets abiertos (sin resolved_at): 240
Tickets críticos                  : 56
Tickets con SLA incumplido        : 95


### Reviso la integridad : ¿todos los eventos tienen su organización y su recurso en los maestros?

Realizo un left join de los eventos comparando contra el maestro: si algún evento no encuentra su organización, la columna del maestro debe quedar en nulo.

In [38]:
orgs_ids = orgs.select("org_id", "org_name")

eventos_sin_org = (
    eventos.select("org_id").distinct()
    .join(orgs_ids, on="org_id", how="left")
    .filter(F.col("org_name").isNull())
)
print("org_id de eventos sin maestro:", eventos_sin_org.count())

org_id de eventos sin maestro: 0


In [39]:
recursos_ids = resources.select("resource_id", "service").withColumnRenamed("service", "service_maestro")

eventos_sin_recurso = (
    eventos.select("resource_id").distinct()
    .join(recursos_ids, on="resource_id", how="left")
    .filter(F.col("service_maestro").isNull())
)
print("resource_id de eventos sin maestro:", eventos_sin_recurso.count())

resource_id de eventos sin maestro: 0


### Rango de fechas de cada fuente

In [40]:
eventos.select(F.min("timestamp"), F.max("timestamp")).show(truncate=False)
billing.select(F.min("month"), F.max("month")).show()
tickets.select(F.min("created_at"), F.max("created_at")).show()

+--------------------+--------------------+
|min(timestamp)      |max(timestamp)      |
+--------------------+--------------------+
|2025-07-03T00:02:00Z|2025-08-31T23:58:00Z|
+--------------------+--------------------+

+----------+----------+
|min(month)|max(month)|
+----------+----------+
|2025-06-01|2025-08-01|
+----------+----------+

+---------------+---------------+
|min(created_at)|max(created_at)|
+---------------+---------------+
|     2025-05-09|     2025-08-31|
+---------------+---------------+



La facturación comienza el mes de junio, pero los eventos recién a partir del 03/07. No puedo comparar uso/facturación en el mes 06. Lo dejo documentado como riesgo.

## 9. Conclusiones

| Hallazgo | Implementar solución |
|---|---|
| value me llega como texto o número | Lo convierto a número en Silver |
| v1 y v2 de los eventos | Unificarlas: en v1, `carbon_kg` y `genai_tokens` quedan nulos |
| unit nulo en algunos eventos | Regla de calidad = quarantine |
| Costos negativos y valores extremos | Regla de calidad + flag de anomalía |
| Eventos desordenados | Watermark + deduplicación por event_id |
| Tipo de cambio de USD distinto de 1 | Forzar a 1 y marcar|
| CSAT y NPS fuera de rango | Marcar como inválidos |
| Facturación desde junio, eventos desde julio | Documentado como riesgo |

El detalle completo está en docs/02_5v_e_inventario_fuentes.md.

In [43]:
# Cerrar la sesión de Spark al terminar
spark.stop()